#### Задача 1
У вас есть база из 4 квартир:

1. **Квартира №1:** Координаты `(40, 10)`, Класс: **«Комфорт»**
2. **Квартира №2:** Координаты `(48, 6)`, Класс: **«Элитное»**
3. **Квартира №3:** Координаты `(55, 12)`, Класс: **«Элитное»**
4. **Квартира №4:** Координаты `(38, 5)`, Класс: **«Комфорт»**

В систему поступает **Новая квартира** с координатами `(50, 8)`.

**Ваша задача:**

1. Посчитайте квадрат расстояния (\(d^{2}\)) от новой квартиры до каждой из четырех в базе.
2. Выберите **3 самых близких** соседа (с наименьшим \(d^{2}\)).
3. Определите по правилу большинства, к какому классу kNN отнесет новую квартиру («Комфорт» или «Элитное»)?

---

#### 1. Мое первое решение

In [2]:
from collections import Counter

k = 3

database = [
    {"coords": [40, 10], "class": "Комфорт"},
    {"coords": [48, 6], "class": "Элитное"},
    {"coords": [55, 12], "class": "Элитное"},
    {"coords": [38, 5], "class": "Комфорт"},
]

new_flat = [50, 8]

closest_neighbor = []
min_squared_dist = float('inf')

for flat in database:
    # Считаем квадрат евклидова расстояние по нашей формуле d^2 = (x1-x2)^2 + (y1-y2)^2
    x_new = new_flat[0]
    y_new = new_flat[1]

    x_train = flat["coords"][0]
    y_train = flat["coords"][1]
    
    d = (x_new - x_train)**2 + (y_new - y_train)**2
    flat['d'] = d

sorted_flat = sorted(database, key=lambda x: x['d'])

# Подходит если всего 2 класса в датасете
# elita = 0
# komfort = 0
# for idx, elem in enumerate(sorted_flat[:k]):
#     if elem['class'] == 'Элитное':
#         elita+=1
#     if elem['class'] == 'Комфорт':
#         komfort+=1

#     if idx == k - 1:
#         if elita >= komfort:
#             new_flat_class = 'Элитное'
#         else:
#             new_flat_class = 'Комфорт'


# Делаем красивее
# 1. Берем только классы первых k соседей
votes = [elem['class'] for elem in sorted_flat[:k]]
# 2. Counter сам считает, сколько раз встретился каждый класс
vote_results = Counter(votes)
# 3. most_common(1) возвращает самый частый элемент, например [('Элитное'), 2]
new_flat_class = vote_results.most_common(1)[0][0]

print('Три ближайших соседа: ', sorted_flat[:k])
print('Класс новой квартиры: ', new_flat_class)


Три ближайших соседа:  [{'coords': [48, 6], 'class': 'Элитное', 'd': 8}, {'coords': [55, 12], 'class': 'Элитное', 'd': 41}, {'coords': [40, 10], 'class': 'Комфорт', 'd': 104}]
Класс новой квартиры:  Элитное


#### Добавим способ взвешивания

Как взвесить голоса соседей (чтобы расстояние 8 имело больший вес, чем 104)?

Вес = **Обратное значение от расстояния**: 1 / d**2

Считаем веса для нашего примера:

1. **Квартира №2** (Элитное, d² = 8): 0.125
2. **Квартира №3** (Элитное, d² = 41): 0.024
3. **Квартира №1** (Комфорт, d² = 104): 0.009

Вместо подсчета количества соседей, складываем их веса по классам:
- сумма весов для "Элитное": 0.125+0.024 = 0.149
- сумма весов для "Комфорт": 0.009

Результат: Класс «Элитное» победил с огромным отрывом 0.149 >> 0.009, так как его представители находятся гораздо ближе.


#### 2. Решение через scikit-learn

В библиотеке `scikit-learn` уже встроен и поиск по деревьям (KD-Tree/Ball-Tree), и взвешивание голосов. За веса отвечает параметр `weights='distance'`. По умолчанию `sklearn` использует обычное евклидово расстояние (с корнем), но для порядка сортировки соседей и итогового взвешивания результат будет математически эквивалентен нашему.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

X_train = np.array([
    [40, 10],
    [48, 6], 
    [55, 12],
    [38, 5],
])

y_train = np.array(["Комфорт", "Элитное", "Элитное", "Комфорт"])

X_new = np.array([[50, 8]])

# weights='distance' -> включаем взвешивание по расстоянию (1 / d)
# metric='minkowski' - по умолчанию, (p=1,2,3,4) p=L - пространства Лебега
# metric='sqeuclidean' -> указываем использовать КВАДРАТ евклидова расстояния
# metric='manhattan' -> для задач: рангов, оценок, прокл. размеров, выбросы, роботы
model = KNeighborsClassifier(n_neighbors=3, weights='distance', metric='sqeuclidean')
# model = KNeighborsClassifier(n_neighbors=3, metric='manhattan') # L1

# 3. Обучаем модель (в случае с kNN она просто запоминает базу данных)
model.fit(X_train, y_train)

predicted_class = model.predict(X_new)[0]
proba = model.predict_proba(X_new)

print(f"🏆 Предсказанный класс через scikit-learn: {predicted_class}")
print(f"📊 Вероятности классов {model.classes_}: {proba[0]}")

🏆 Предсказанный класс через scikit-learn: Элитное
📊 Вероятности классов ['Комфорт' 'Элитное']: [0.06047198 0.93952802]


🛠️ Золотые правила выбора \(k\):

1. **Выбирайте нечетное число:** Если у вас 2 класса, выбирайте \(k = 3, 5, 7...\). Это исключит ситуацию «ничьей», когда голоса разделились строго поровну (например, 2 на 2 при \(k=4\)).
2. **Эмпирическое правило (базовый ориентир):** Хорошей стартовой точкой считается корень из общего количества объектов в обучающей выборке **k(sqrt){N}**  
    _Если у вас в базе 100 квартир, попробуйте \(k = 10\) (или лучше 9 / 11, чтобы число было нечетным)._
3. **Кросс-валидация (GridSearchCV) — метод Data Science:** На практике запускают цикл, который проверяет работу kNN при разных значениях \(k\) (от 1 до 30) на тестовых данных, и выбирает то \(k\), при котором модель совершает меньше всего ошибок.

---

#### Для регрессии

In [10]:
from sklearn.neighbors import KNeighborsRegressor

# Создаем регрессор (метрику берем нашу проверенную - квадрат евклидова)
regressor = KNeighborsRegressor(n_neighbors=3, weights='distance', metric='sqeuclidean')

# Обучаем на координатах (например, площадь и центр) и реальных ценах в млн.
X_train = [[40, 10], [48, 6], [55, 12]]
y_train = [6.0, 7.0, 11.0] # Цены

regressor.fit(X_train, y_train)
# Предскажет точную стоимость для новой квартиры
predicted_price = regressor.predict([[50, 8]]) 
print(f"{predicted_price[0]:.2f} млн")


7.55 млн
